# Code to download and preprocess REMA strips over ice shelves

In [1]:
project_crs = "EPSG:3031"

projectDir = "/Users/b513882/Library/CloudStorage/Dropbox/REMA-strips-correction"

# This is just a name that you give your region
# This will be used when saving files and creating folders
region_id = "PIG" 

In [2]:
import geopandas as gpd
import preprocessing_tools as ptools

## Define an Area Of Interest, and call that region something

Here I have called the region "PIG" which we will use througout the code

In [3]:

# Coordinates of Area Of Interest (AOI)
# coords = [
#     (-102.53687728768465, -75.49231807240791),
#     (-100.47694076424715, -75.49231807240791),
#     (-100.47694076424715, -75.02383074354489),
#     (-102.53687728768465, -75.02383074354489),
#     (-102.53687728768465, -75.49231807240791)
# ]

coords = [(-101.88705298569928, -75.2944911579537),
(-101.19491431382428, -75.2944911579537),
(-101.19491431382428, -75.06688632944639),
(-101.88705298569928, -75.06688632944639),
(-101.88705298569928, -75.2944911579537)
]

# CRS of coordinates
coords_crs = "EPSG:4326"

region = ptools.polygon_coords_to_gdf(coords,coords_crs,project_crs,projectDir,region_id)



## Download the REMA strips

Before downloading the REMA strips you need to download the REMA Strip DEM extent index SHP file from https://www.pgc.umn.edu/data/rema/

Make sure that the shapefile gets stored here: `{projectDir}/data/initial/REMA/REMA_Strip_Index_s2s041.shp`

Now you can run the download script like this if you want to keep the original resolution:

`python download-REMA-strips.py --region_id PIG --projectDir "/path/to/project" --resample false`

Or like this if you want to resample to for instance 20 m resolution:

`python download-REMA-strips.py --region_id PIG --projectDir "/path/to/project" --resample true --resolution 20`


If you're on MacOS and want to prevent your laptop from sleeping while downloading, you can use:

`caffeinate -i python download-REMA-strips.py --region_id PIG --projectDir "/path/to/project" --resample false`

Here is the complete list of flags:

`--region_id PIG`: The name of the region/AOI

`--project_crs EPSG:3031`: CRS of the project, EPSG:3031 is the default

`--n_threads 8`: Number of parallel threads for the download, default is 8

`--projectDir "/path/to/project"`: path to project directory

`--resample true`: whether or not to resample the strips

`--resolution 20`: Resample resolution in meters if resampling is chosen

## Bit masking

Cleaning the strips based on the provided bitmask

In [3]:
# Bit masking

n_threads = 8 # Number of CPU cores to use

ptools.bit_masking(region_id,projectDir,n_threads)

Finished bit masking for region PIG


## Tide correction

The tides are being calculated on a 1x1 km grid and interpolated onto the REMA grid. The original resolution of the tide model is 2 km.

If you don't already have CATS2008_v2023, you first need to it. You can download it here: https://www.usap-dc.org/view/dataset/601772

I have chosen to save CATS in data/initial, but you can choose where you save it and just point to the right directory below with `tide_model_dir`.

In [4]:
ptools.tide_correction(
    region_id="PIG",
    projectDir=projectDir,
    tide_model_dir=f"{projectDir}/data/initial",
    tide_resolution=1000,
    tide_buffer=5000,
    save_tide=True,
) 


Found 121 masked REMA strips for region PIG
CATS tide grid resolution: 1000 m

Strip 1/121

Processing: SETSM_s2s041_W1W1_20110211_1020010011D98C00_1020010012104600_2m_lsf_seg2_dem_10m.tif
   Acquisition 1: 2011-02-11 15:34:56
   Acquisition 2: 2011-02-19 15:46:38
   Tide grid: 30 × 29 pixels at 1000 m
   Mean tide range: -0.20 to -0.18 m
   Interpolating tide onto REMA grid...
   Saving tide-corrected DEM...
✅ Finished: SETSM_s2s041_W1W1_20110211_1020010011D98C00_1020010012104600_2m_lsf_seg2_dem_10m.tif

Strip 2/121

Processing: SETSM_s2s041_W1W1_20110924_1020010015D51700_1020010016812100_2m_lsf_seg2_dem_10m.tif
   Acquisition 1: 2011-10-06 15:58:51
   Acquisition 2: 2011-09-24 15:42:00
   Tide grid: 25 × 32 pixels at 1000 m
   Mean tide range: 0.59 to 0.61 m
   Interpolating tide onto REMA grid...
   Saving tide-corrected DEM...
✅ Finished: SETSM_s2s041_W1W1_20110924_1020010015D51700_1020010016812100_2m_lsf_seg2_dem_10m.tif

Strip 3/121

Processing: SETSM_s2s041_W1W1_20111006_102001

## Co-registration

The tide-corrected strips are co-registered to CryoSat-2 SWATH and POCA measurements.

In [7]:
qc = ptools.coregistration(
    region_id=region_id,
    projectDir=projectDir,
    cores=3,
    qc_resolution_m=2000,
    poca_total_weight=5.0,
    swath_total_weight=1.0,
    second_pass_threshold_m=15.0,
    min_points_for_plane=10,
    min_poca=10,
    min_swath=20,
    min_footprint_coverage=0.20,
    min_covered_quadrants=3,
    max_rmse_m=10.0,
    max_rejected_fraction=0.5,
    overwrite=True,
)

Co-registering 119 REMA strips for PIG using 3 processes
[1/119] SETSM_s2s041_W1W1_20110924_1020010015D51700_1020010016812100_2m_lsf_seg2_dem_10m.tif: FLAGGED RMSE 74.27 -> 7.82 m (LOW_FOOTPRINT_COVERAGE)
[2/119] SETSM_s2s041_W1W1_20110211_1020010011D98C00_1020010012104600_2m_lsf_seg2_dem_10m.tif: OK RMSE 36.20 -> 6.65 m ()
[3/119] SETSM_s2s041_W1W1_20120112_1020010018793000_1020010018CE7400_2m_lsf_seg3_dem_10m.tif: UNCORRECTED RMSE nan -> nan m (LOW_POCA_COUNT;LOW_SWATH_COUNT;LOW_FOOTPRINT_COVERAGE;POOR_SPATIAL_DISTRIBUTION;NO_REFERENCE_OBSERVATIONS)
[4/119] SETSM_s2s041_W1W1_20120112_1020010018793000_1020010018CE7400_2m_lsf_seg4_dem_10m.tif: UNCORRECTED RMSE nan -> nan m (LOW_POCA_COUNT;LOW_SWATH_COUNT;LOW_FOOTPRINT_COVERAGE;POOR_SPATIAL_DISTRIBUTION;NO_REFERENCE_OBSERVATIONS)
[5/119] SETSM_s2s041_W1W1_20111006_1020010015077900_1020010015D51700_2m_lsf_seg1_dem_10m.tif: OK RMSE 7.15 -> 5.96 m ()
[6/119] SETSM_s2s041_W1W1_20111006_1020010015A6C400_1020010015D51700_2m_lsf_seg1_dem_10m.t

/Users/b513882/Library/CloudStorage/Dropbox/REMA-strips-correction/preprocessing_tools.py:992: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data = pd.concat([poca, swath], ignore_index=True)


[10/119] SETSM_s2s041_W2W2_20101116_10300100076B4400_1030010007D70D00_2m_lsf_seg1_dem_10m.tif: OK RMSE 15.72 -> 8.08 m ()
[11/119] SETSM_s2s041_W1W2_20110304_1020010011431700_1030010009514D00_2m_lsf_seg1_dem_10m.tif: OK RMSE 11.45 -> 6.47 m ()
[12/119] SETSM_s2s041_W1W2_20110315_1020010012B61800_10300100091A4B00_2m_lsf_seg1_dem_10m.tif: OK RMSE 8.86 -> 6.61 m ()
[13/119] SETSM_s2s041_W2W2_20101124_1030010007D70D00_103001000842B700_2m_lsf_seg4_dem_10m.tif: FLAGGED RMSE 120.18 -> 109.17 m (LOW_POCA_COUNT;HIGH_RMSE)
[14/119] SETSM_s2s041_W2W2_20101116_10300100076B4400_1030010008691800_2m_lsf_seg2_dem_10m.tif: FLAGGED RMSE 53.43 -> 17.21 m (HIGH_RMSE)
[15/119] SETSM_s2s041_W2W2_20101124_1030010007D70D00_103001000842B700_2m_lsf_seg2_dem_10m.tif: FLAGGED RMSE 41.65 -> 38.04 m (HIGH_RMSE)
[16/119] SETSM_s2s041_W2W2_20101124_1030010007D70D00_1030010008691800_2m_lsf_seg3_dem_10m.tif: OK RMSE 25.62 -> 9.13 m ()
[17/119] SETSM_s2s041_W2W2_20101130_103001000771AA00_1030010007B56A00_2m_lsf_seg2_dem

/Users/b513882/Library/CloudStorage/Dropbox/REMA-strips-correction/preprocessing_tools.py:992: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data = pd.concat([poca, swath], ignore_index=True)


[26/119] SETSM_s2s041_W2W2_20101216_1030010007B56A00_1030010008833C00_2m_lsf_seg4_dem_10m.tif: FLAGGED RMSE 36.74 -> 25.19 m (HIGH_RMSE)
[27/119] SETSM_s2s041_W2W2_20101230_1030010008399100_1030010008C0DA00_2m_lsf_seg1_dem_10m.tif: FLAGGED RMSE 67.70 -> 12.26 m (HIGH_RMSE)
[28/119] SETSM_s2s041_W2W2_20101230_1030010008399100_1030010008476F00_2m_lsf_seg1_dem_10m.tif: FLAGGED RMSE 69.33 -> 15.38 m (HIGH_RMSE)
[29/119] SETSM_s2s041_W2W2_20111222_103001000F578C00_1030010010AD2800_2m_lsf_seg5_dem_10m.tif: FLAGGED RMSE 50.97 -> 47.27 m (LOW_FOOTPRINT_COVERAGE;HIGH_RMSE)
[30/119] SETSM_s2s041_W2W2_20120127_1030010010333800_1030010011B36500_2m_lsf_seg2_dem_10m.tif: UNCORRECTED RMSE nan -> nan m (LOW_POCA_COUNT;LOW_SWATH_COUNT;LOW_FOOTPRINT_COVERAGE;POOR_SPATIAL_DISTRIBUTION;NO_REFERENCE_OBSERVATIONS)
[31/119] SETSM_s2s041_W2W2_20120127_1030010011B8D700_1030010011C3CB00_2m_lsf_seg2_dem_10m.tif: UNCORRECTED RMSE nan -> nan m (LOW_POCA_COUNT;LOW_SWATH_COUNT;LOW_FOOTPRINT_COVERAGE;POOR_SPATIAL_DIS

In [8]:
qc[qc['status']=="OK"]

,strip,status,quality_flags,error,acquisition_time_utc,months,n_poca,n_swath,n_total,valid_footprint_km2,...,n_rejected,rmse_before,rmse_after,weighted_rmse_before,weighted_rmse_after,rmse_poca_before,rmse_poca_after,rmse_swath_before,rmse_swath_after,output_file
1,SETSM_s2s041_W1W1_20110211_1020010011D98C00_10...,OK,,,2011-02-15T15:40:47+00:00,2011-01;2011-02,45,477,522,333.3168,...,22.0,36.202750,6.654797,32.573068,5.150758,31.706736,4.750721,36.598403,6.806988,/Users/b513882/Library/CloudStorage/Dropbox/RE...
4,SETSM_s2s041_W1W1_20111006_1020010015077900_10...,OK,,,2011-10-06T15:59:03+00:00,2011-09;2011-10,164,1088,1252,851.7240,...,30.0,7.148448,5.964784,4.408204,3.709203,3.452158,2.929193,7.550269,6.296687,/Users/b513882/Library/CloudStorage/Dropbox/RE...
9,SETSM_s2s041_W2W2_20101116_10300100076B4400_10...,OK,,,2010-11-20T15:56:16+00:00,2010-11;2010-12,25,670,695,289.9776,...,56.0,15.716762,8.081138,10.682857,9.508704,9.292241,9.780805,15.906345,8.010741,/Users/b513882/Library/CloudStorage/Dropbox/RE...
10,SETSM_s2s041_W1W2_20110304_1020010011431700_10...,OK,,,2011-03-04T15:34:40+00:00,2011-02;2011-03,479,2348,2827,951.2195,...,100.0,11.450951,6.466925,8.349685,5.290541,7.368949,4.951522,12.115956,6.734323,/Users/b513882/Library/CloudStorage/Dropbox/RE...
11,SETSM_s2s041_W1W2_20110315_1020010012B61800_10...,OK,,,2011-03-15T16:41:02+00:00,2011-02;2011-03,340,1404,1744,890.1775,...,58.0,8.864961,6.611109,6.511419,5.639144,5.740248,5.356430,9.467801,6.880623,/Users/b513882/Library/CloudStorage/Dropbox/RE...
15,SETSM_s2s041_W2W2_20101124_1030010007D70D00_10...,OK,,,2010-11-27T15:52:34+00:00,2010-11;2010-12,59,626,685,264.9264,...,62.0,25.623712,9.134521,20.986050,4.826649,19.804009,3.143470,26.105398,9.506433,/Users/b513882/Library/CloudStorage/Dropbox/RE...
22,SETSM_s2s041_W2W2_20101208_1030010008064E00_10...,OK,,,2010-12-10T19:47:33+00:00,2010-11;2010-12,93,404,497,214.8852,...,0.0,13.271261,4.144964,11.890671,2.730602,11.507760,2.224431,13.644982,4.471762,/Users/b513882/Library/CloudStorage/Dropbox/RE...
24,SETSM_s2s041_W2W2_20101208_10300100089AF900_10...,OK,,,2010-12-10T19:47:25+00:00,2010-11;2010-12,90,325,415,355.2765,...,30.0,21.555887,8.182166,20.575526,5.872958,20.302349,5.071176,21.890333,8.852438,/Users/b513882/Library/CloudStorage/Dropbox/RE...
31,SETSM_s2s041_W2W2_20111222_103001000F578C00_10...,OK,,,2011-12-27T04:13:31+00:00,2011-12;2012-01,42,86,128,94.8752,...,5.0,27.380979,8.907184,25.685900,4.885063,25.101601,2.358625,28.427817,10.740926,/Users/b513882/Library/CloudStorage/Dropbox/RE...
37,SETSM_s2s041_WV01_20120114_10200100174FFB00_10...,OK,,,2012-01-14T15:11:28+00:00,2011-12;2012-01,391,3965,4356,4796.0945,...,146.0,7.601227,7.521270,4.586028,4.521665,3.577019,3.514041,7.887624,7.805784,/Users/b513882/Library/CloudStorage/Dropbox/RE...
